# 环境配置
- conda create -n teach python=3.10
- conda activate teach

# 8数码难题

**教学说明（本科生）：**

8数码难题是人工智能搜索算法教学中的经典问题。本节我们将学习如何判断问题是否有解，并使用不同的搜索算法求解。

**问题变体：**
- **3×3棋盘**（8数码）：比15数码更简单，适合初学者理解搜索算法
- **可解性判定**：只需比较初始状态和目标状态的逆序数奇偶性

**学习重点：**
1. 理解逆序数在可解性判定中的作用
2. 为后续学习 BFS、DFS、A* 等搜索算法做准备
3. 体会不同搜索算法的性能差异

---
### 📚 学习重点

- 理解8数码问题与15数码问题的关系
- 掌握逆序数在可解性判定中的统一原理
- 对比BFS、DFS、A*三种搜索算法的特点

In [ ]:
def inversion_count(board):
    nums = [x for x in board if x != 0]
    inv = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv += 1
    return inv

def is_solvable_3x3(start, goal):
    return inversion_count(start) % 2 == inversion_count(goal) % 2


# 例子：0 表示空格
start = [
    2, 8, 3,
    1, 0, 4,
    7, 6, 5
]

goal = [
    1, 2, 3,
    8, 0, 4,
    7, 6, 5
]

print("初始逆序数 =", inversion_count(start))
print("目标逆序数 =", inversion_count(goal))

if is_solvable_3x3(start, goal):
    print("有解")
else:
    print("无解")

## 宽度优先搜索

**教学说明（本科生）：**

宽度优先搜索（BFS, Breadth-First Search）是一种系统性的搜索算法，它按层次遍历状态空间。

**核心特点：**
- **层次遍历**：先访问距离起点近的状态，再访问远的状态
- **最优性**：如果每个动作的代价相同，BFS能找到最短路径
- **完备性**：只要解存在，BFS一定能找到

**算法思想：**
使用队列（queue）数据结构存储待访问的状态。每次从队列头部取出状态进行扩展，将新状态加入队列尾部。

**应用场景：**
- 迷宫求解
- 社交网络中的最短联系路径
- 任务规划中的最优动作序列

---
### 📚 学习重点

- 理解8数码问题与15数码问题的关系
- 掌握逆序数在可解性判定中的统一原理
- 对比BFS、DFS、A*三种搜索算法的特点

### BFS vs DFS vs A* 对比

| 特性 | BFS | DFS | A* |
|------|-----|-----|-----|
| 数据结构 | 队列 (Queue) | 栈 (Stack) | 优先队列 (Priority Queue) |
| 搜索策略 | 广度优先 | 深度优先 | 启发式优先 |
| 最优性 | ✅ 保证最短路径 | ❌ 不保证最优 | ✅ 保证最优（h可采纳时） |
| 空间复杂度 | O(b^d) 高 | O(bd) 低 | O(b^d) 中等 |
| 完备性 | ✅ 有解必找到 | ❌ 可能无限循环 | ✅ 有解必找到 |

> 💡 **教学提示**：可以把BFS想象成"水波扩散"——从投石点一圈圈向外推开，最先到达的就是最近的路。

In [ ]:
from collections import deque, defaultdict

def inversion_count(board):
    nums = [x for x in board if x != 0]
    inv = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv += 1
    return inv

def is_solvable_3x3(start, goal):
    return inversion_count(start) % 2 == inversion_count(goal) % 2

def get_neighbors(state):
    """
    生成当前状态的所有相邻状态
    返回 [(next_state, action), ...]
    action 表示空格移动方向
    """
    neighbors = []
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)

    moves = [
        (-1, 0, "空格上移"),
        (1, 0, "空格下移"),
        (0, -1, "空格左移"),
        (0, 1, "空格右移")
    ]

    for dr, dc, action in moves:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            new_state = list(state)
            new_state[zero_idx], new_state[new_idx] = new_state[new_idx], new_state[zero_idx]
            neighbors.append((tuple(new_state), action))

    return neighbors

def bfs_all_shortest_paths(start, goal):
    """
    宽度优先搜索 BFS：返回所有最短路径
    每条路径格式：
    [(state0, "初始状态"), (state1, action1), ..., (goal, actionN)]
    """
    start = tuple(start)
    goal = tuple(goal)

    queue = deque([start])
    dist = {start: 0}
    parents = defaultdict(list)   # state -> [(parent_state, action), ...]

    while queue:
        current = queue.popleft()
        current_dist = dist[current]

        for next_state, action in get_neighbors(current):
            # 第一次到达 next_state
            if next_state not in dist:
                dist[next_state] = current_dist + 1
                parents[next_state].append((current, action))
                queue.append(next_state)
            # 又找到一条同样短的路径
            elif dist[next_state] == current_dist + 1:
                parents[next_state].append((current, action))

    if goal not in dist:
        return None

    # 回溯所有最短路径
    all_paths = []

    def backtrack(state, path):
        # path 反向存储 [(state, action_from_parent), ...]
        if state == start:
            full_path = [(start, "初始状态")] + list(reversed(path))
            all_paths.append(full_path)
            return

        for parent_state, action in parents[state]:
            backtrack(parent_state, path + [(state, action)])

    backtrack(goal, [])
    return all_paths

def print_board(state):
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join("□" if x == 0 else str(x) for x in row))
    print()

# =========================
# 输入初始状态和目标状态
# =========================
start = [
    2, 8, 3,
    1, 0, 4,
    7, 6, 5
]

goal = [
    1, 2, 3,
    8, 0, 4,
    7, 6, 5
]

# =========================
# 输出结果
# =========================
print("初始逆序数 =", inversion_count(start))
print("目标逆序数 =", inversion_count(goal))

if is_solvable_3x3(start, goal):
    print("结论：有解\n")

    all_paths = bfs_all_shortest_paths(start, goal)

    if all_paths:
        print("宽度优先搜索（BFS）找到的最短步数 =", len(all_paths[0]) - 1)
        print("最短解路径条数 =", len(all_paths))
        print()

        # 打印所有最短路径
        for path_idx, path in enumerate(all_paths, 1):
            print(f"================ 第 {path_idx} 条最短路径 ================\n")
            for step, (state, action) in enumerate(path):
                print(f"第 {step} 步：{action}")
                print_board(state)

        # 打印最佳那一条（这里取第一条最短路径）
        best_path = all_paths[0]
        print("============== 最佳路径（取第一条最短路径） ==============\n")
        for step, (state, action) in enumerate(best_path):
            print(f"第 {step} 步：{action}")
            print_board(state)

    else:
        print("BFS 未找到解")
else:
    print("结论：无解")

## 深度优先搜索

**教学说明（本科生）：**

深度优先搜索（DFS, Depth-First Search）沿着搜索树的分支尽可能深地搜索。

**核心特点：**
- **深度优先**：沿着一条路径一直走到底，无法继续时回溯
- **空间效率高**：只需存储从起点到当前状态的路径
- **不一定最优**：找到的第一个解不一定是最佳解

**算法思想：**
使用栈（stack）或递归实现。每次选择一个未访问的邻接状态继续搜索，当到达终点或无法继续时回溯。

**应用场景：**
- 路径存在性判断
- 拓扑排序
- 迷宫求解（找到任意解即可的场景）

---
### 📚 学习重点

- 理解8数码问题与15数码问题的关系
- 掌握逆序数在可解性判定中的统一原理
- 对比BFS、DFS、A*三种搜索算法的特点

### 深度限制的重要性

DFS如果不加深度限制，可能会沿着一条错误的路径无限搜索下去。这就是为什么代码中使用了 `depth_limit` 参数。

> 💡 **教学提示**：DFS就像"钻进一条胡同走到底"，如果走不通再退回来换一条。优点是内存占用小，缺点是不一定找到最短路径。

In [ ]:
def inversion_count(board):
    nums = [x for x in board if x != 0]
    inv = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv += 1
    return inv

def is_solvable_3x3(start, goal):
    return inversion_count(start) % 2 == inversion_count(goal) % 2

def get_neighbors(state):
    """
    生成当前状态的所有相邻状态
    返回 [(next_state, action), ...]
    action 表示空格移动方向
    """
    neighbors = []
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)

    moves = [
        (-1, 0, "空格上移"),
        (1, 0, "空格下移"),
        (0, -1, "空格左移"),
        (0, 1, "空格右移")
    ]

    for dr, dc, action in moves:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            new_state = list(state)
            new_state[zero_idx], new_state[new_idx] = new_state[new_idx], new_state[zero_idx]
            neighbors.append((tuple(new_state), action))

    return neighbors

def dfs_limited(start, goal, depth_limit):
    """
    深度优先搜索（带深度约束）
    返回一条路径：
    [(state0, "初始状态"), (state1, action1), ..., (goal, actionN)]
    """
    start = tuple(start)
    goal = tuple(goal)

    path = [(start, "初始状态")]
    visited_in_path = {start}

    def dfs(state, depth):
        if state == goal:
            return True

        if depth == depth_limit:
            return False

        for next_state, action in get_neighbors(state):
            if next_state not in visited_in_path:
                visited_in_path.add(next_state)
                path.append((next_state, action))

                if dfs(next_state, depth + 1):
                    return True

                path.pop()
                visited_in_path.remove(next_state)

        return False

    found = dfs(start, 0)
    return path if found else None

def print_board(state):
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join("□" if x == 0 else str(x) for x in row))
    print()

# =========================
# 输入初始状态和目标状态
# =========================
start = [
    2, 8, 3,
    1, 0, 4,
    7, 6, 5
]

goal = [
    1, 2, 3,
    8, 0, 4,
    7, 6, 5
]

depth_limit = 10   # 这里可以改成 4、6、8、10 等

# =========================
# 输出结果
# =========================
print("初始逆序数 =", inversion_count(start))
print("目标逆序数 =", inversion_count(goal))

if is_solvable_3x3(start, goal):
    print("结论：有解\n")

    path = dfs_limited(start, goal, depth_limit)

    if path:
        print(f"深度优先搜索（DFS）在深度约束 = {depth_limit} 下找到一条解路径")
        print("路径步数 =", len(path) - 1)
        print()

        for step, (state, action) in enumerate(path):
            print(f"第 {step} 步：{action}")
            print_board(state)
    else:
        print(f"在深度约束 = {depth_limit} 下，DFS 未找到解")
else:
    print("结论：无解")

## A*算法

**教学说明（本科生）：**

A*（A-Star）算法是**启发式搜索**中最经典、最重要的算法，结合了最佳优先搜索和Dijkstra算法的思想。

**核心思想：**
使用评估函数 f(n) = g(n) + h(n) 来指导搜索方向：
- **g(n)**：从起点到当前节点的实际代价
- **h(n)**：从当前节点到目标节点的启发式估计（heuristic）
- **f(n)**：从起点经过当前节点到目标的总估计代价

**启发式函数：**
- **曼哈顿距离**：|x1-x2| + |y1-y2|，适用于只能上下左右移动的场景
- **欧几里得距离**：两点间直线距离
- **对角距离**：适用于允许对角移动的场景

**算法优势：**
- 如果启发式函数是**可采纳的**（h(n) ≤ 实际代价），A*能找到最优解
- 相比BFS，A*用更少的节点扩展找到最优解，效率更高

**应用场景：**
- 地图寻路（如Google Maps）
- 游戏NPC寻路
- 机器人路径规划

---
### 📚 学习重点

- 理解8数码问题与15数码问题的关系
- 掌握逆序数在可解性判定中的统一原理
- 对比BFS、DFS、A*三种搜索算法的特点

In [ ]:
import heapq

# =========================
# 八数码 A* 算法
# =========================

def print_board(state):
    """打印棋盘"""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join(str(x) if x != 0 else " " for x in row))
    print()


def manhattan_distance(state, goal):
    """曼哈顿距离启发函数 h(n)"""
    distance = 0
    goal_pos = {}

    for i, value in enumerate(goal):
        goal_pos[value] = (i // 3, i % 3)

    for i, value in enumerate(state):
        if value != 0:
            x1, y1 = i // 3, i % 3
            x2, y2 = goal_pos[value]
            distance += abs(x1 - x2) + abs(y1 - y2)

    return distance


def get_neighbors(state):
    """生成当前状态的所有相邻状态"""
    neighbors = []
    zero_index = state.index(0)
    x, y = zero_index // 3, zero_index % 3

    moves = [
        (-1, 0, "上"),
        (1, 0, "下"),
        (0, -1, "左"),
        (0, 1, "右")
    ]

    for dx, dy, action in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_index = nx * 3 + ny
            new_state = list(state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
            neighbors.append((tuple(new_state), action))

    return neighbors


def reconstruct_path(came_from, move_from, current):
    """回溯路径"""
    path = []
    moves = []

    while current in came_from:
        path.append(current)
        moves.append(move_from[current])
        current = came_from[current]

    path.append(current)   # 加入初始状态
    path.reverse()
    moves.reverse()

    return path, moves


def inversion_count(state):
    """计算逆序数，用于判断是否有解"""
    arr = [x for x in state if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv


def is_solvable(start, goal):
    """3x3 八数码可解判定：逆序奇偶性相同才可解"""
    return inversion_count(start) % 2 == inversion_count(goal) % 2


def a_star(start, goal):
    """A* 搜索"""
    if not is_solvable(start, goal):
        return None, None

    open_heap = []
    g_score = {start: 0}
    f_score = {start: manhattan_distance(start, goal)}
    came_from = {}
    move_from = {start: None}

    # 堆中元素: (f, g, state)
    heapq.heappush(open_heap, (f_score[start], 0, start))

    visited = set()

    while open_heap:
        current_f, current_g, current = heapq.heappop(open_heap)

        if current in visited:
            continue
        visited.add(current)

        if current == goal:
            return reconstruct_path(came_from, move_from, current)

        for neighbor, action in get_neighbors(current):
            tentative_g = g_score[current] + 1

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                move_from[neighbor] = action
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + manhattan_distance(neighbor, goal)
                heapq.heappush(open_heap, (f_score[neighbor], tentative_g, neighbor))

    return None, None


if __name__ == "__main__":
    # 图中的初始状态和目标状态
    start = (2, 8, 3,
             1, 6, 4,
             7, 0, 5)

    goal = (1, 2, 3,
            8, 0, 4,
            7, 6, 5)

    print("初始状态：")
    print_board(start)

    print("目标状态：")
    print_board(goal)

    path, moves = a_star(start, goal)

    if path is None:
        print("该八数码无解。")
    else:
        print(f"找到解，共 {len(path) - 1} 步。\n")

        for i, state in enumerate(path):
            if i == 0:
                print(f"第 {i} 步：初始状态")
            else:
                print(f"第 {i} 步：空格向{moves[i-1]}移动")
            print_board(state)

# 相关资料
- 搜索：https://www.redblobgames.com/pathfinding/a-star/introduction.html 
- 寻路：https://abelchiao.github.io/genetic-algorithm-visualization/

### 拓展阅读

- 交互式A*可视化：通过图形界面直观理解搜索过程
- 遗传算法可视化：观察进化过程如何优化路径